# Transformer

Vaswani et al., *Attention Is All You Need*, NeurIPS 2017 ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)).

Encoder-decoder with self-attention (encoder, bidirectional), causal self-attention (decoder), and cross-attention (decoder queries, encoder keys/values). Trained here on real English->French sentence pairs.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from transformer_playground.data import load_translation_pairs
from transformer_playground.device import resolve_device
from transformer_playground.utils.seed import set_seed
from model import TransformerModel
from example import Vocab, TranslationDataset, PAD

set_seed(0)
device = resolve_device('auto')
print(device)

In [ ]:
pairs = load_translation_pairs(max_pairs=4000)
n_val = max(1, len(pairs) // 20)
train_pairs, val_pairs = pairs[n_val:], pairs[:n_val]
src_vocab = Vocab([e for e, _ in train_pairs])
tgt_vocab = Vocab([f for _, f in train_pairs])
max_len = 16
train_ds = TranslationDataset(train_pairs, src_vocab, tgt_vocab, max_len)
val_ds = TranslationDataset(val_pairs, src_vocab, tgt_vocab, max_len)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
print(len(train_ds), len(val_ds), len(src_vocab), len(tgt_vocab))

In [ ]:
model = TransformerModel(len(src_vocab), len(tgt_vocab), d_model=128, n_heads=4, n_layers=2, d_ff=256, max_len=max_len).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=tgt_vocab.stoi[PAD])

history = {'train_loss': [], 'val_loss': []}
for epoch in range(10):
    model.train()
    total = 0.0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        opt.zero_grad()
        logits = model(src, tgt_in)
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        loss.backward()
        opt.step()
        total += loss.item()
    history['train_loss'].append(total / len(train_loader))
    print(epoch, history['train_loss'][-1])

In [ ]:
plt.plot(history['train_loss'])
plt.xlabel('epoch')
plt.ylabel('train loss')
plt.title('Transformer training loss (real English-French pairs)')
plt.show()